# 04 — what the archive is missing, and why

Every hole the lake knows about is a row or a flag somewhere: trade-id gaps in silver, venue replays, Kraken checksum failures by hour, and the acknowledged offset gaps and chaos windows in `audit.checks`. This notebook reads them all.

In [1]:
from k2lake import connect, pin
con = connect()
PIN = pin(con)   # every query below reads these snapshot ids, never the moving head
con.sql("""
SELECT 'binance' AS venue, count(*) AS rows, sum(venue_replay::INT) AS replays, sum(seq_gap::INT) AS gaps, sum(missing_before) AS ids_never_received FROM pinned.silver_trades_binance
UNION ALL SELECT 'kraken', count(*), sum(venue_replay::INT), sum(seq_gap::INT), sum(missing_before) FROM pinned.silver_trades_kraken
UNION ALL SELECT 'coinbase', count(*), sum(venue_replay::INT), sum(seq_gap::INT), sum(missing_before) FROM pinned.silver_trades_coinbase
""").show()

pinned 18 tables at commit b12cff9
  audit.checks                 4558366752866520299
  gold.bars                    1589136844755948459
  gold.bbo_1s                  1578478823608857678
  gold.book_state              6881440282983849882
  gold.book_top20              1206037850593976770
  gold.dim_instrument          1768279322099994693
  gold.dim_venue               4191380317728821135
  gold.ohlcv_1d                1579946688158752964
  gold.ohlcv_1h                2397688695969945394
  gold.ohlcv_1m                1622213366608023449
  gold.ohlcv_5m                1666460790140246036
  gold.trades                  1348216816157062508
  silver.book_binance          7922581822436893978
  silver.book_coinbase         5196050954894723418
  silver.book_kraken           9210612890945089666
  silver.trades_binance        3528392281922118482
  silver.trades_coinbase       7144589879523206615
  silver.trades_kraken         264358488000747777
┌──────────┬─────────┬─────────┬────────┬───────

When were trades missed? Gaps by hour, per venue — a capture restart, a produce-error drop or a retention eviction each leave a signature.

In [2]:
con.sql("""
WITH g AS (
  SELECT 'binance' AS venue, date_trunc('hour', exchange_ts) AS h, missing_before FROM pinned.silver_trades_binance WHERE seq_gap
  UNION ALL SELECT 'kraken', date_trunc('hour', exchange_ts), missing_before FROM pinned.silver_trades_kraken WHERE seq_gap
  UNION ALL SELECT 'coinbase', date_trunc('hour', exchange_ts), missing_before FROM pinned.silver_trades_coinbase WHERE seq_gap)
SELECT h, venue, count(*) AS gaps, sum(missing_before) AS ids FROM g GROUP BY 1, 2 ORDER BY 1, 2
""").show(max_rows=40)

┌──────────────────────────┬──────────┬───────┬────────┐
│            h             │  venue   │ gaps  │  ids   │
│ timestamp with time zone │ varchar  │ int64 │ int128 │
├──────────────────────────┼──────────┼───────┼────────┤
│ 2026-08-26 12:00:00+00   │ binance  │     8 │    533 │
│ 2026-08-26 12:00:00+00   │ coinbase │     1 │     25 │
│ 2026-08-26 12:00:00+00   │ kraken   │    29 │    516 │
│ 2026-08-26 13:00:00+00   │ binance  │    20 │   2333 │
│ 2026-08-26 13:00:00+00   │ kraken   │     6 │     32 │
│ 2026-08-26 14:00:00+00   │ binance  │     9 │   1032 │
│ 2026-08-26 14:00:00+00   │ kraken   │     4 │   3543 │
│ 2026-08-26 15:00:00+00   │ kraken   │     9 │     10 │
│ 2026-08-26 16:00:00+00   │ binance  │    47 │ 113764 │
│ 2026-08-26 16:00:00+00   │ coinbase │    36 │  27004 │
│ 2026-08-26 16:00:00+00   │ kraken   │    45 │   1953 │
│ 2026-08-26 17:00:00+00   │ binance  │    17 │  76424 │
│ 2026-08-26 17:00:00+00   │ coinbase │    12 │  14065 │
│ 2026-08-26 17:00:00+00   │ kr

Kraken book integrity, per hour: frames whose replayed book hashed to the venue's checksum, failed it, or could not be checked (no snapshot in the archive for that connection).

In [3]:
con.sql("""
SELECT date_trunc('hour', recv_ts) AS h, sum(CASE WHEN checksum_ok THEN 1 ELSE 0 END) AS verified,
       sum(CASE WHEN checksum_ok = false THEN 1 ELSE 0 END) AS failed, sum(CASE WHEN checksum_ok IS NULL THEN 1 ELSE 0 END) AS unverifiable
FROM pinned.silver_book_kraken GROUP BY 1 ORDER BY 1
""").show(max_rows=48)

┌──────────────────────────┬──────────┬────────┬──────────────┐
│            h             │ verified │ failed │ unverifiable │
│ timestamp with time zone │  int128  │ int128 │    int128    │
├──────────────────────────┼──────────┼────────┼──────────────┤
│ 2026-08-26 12:00:00+00   │  1751460 │      0 │            0 │
│ 2026-08-26 13:00:00+00   │  2901956 │      0 │        82738 │
│ 2026-08-26 14:00:00+00   │  2604444 │      0 │       406686 │
│ 2026-08-26 15:00:00+00   │  3246261 │      0 │        42667 │
│ 2026-08-26 16:00:00+00   │  1886540 │ 153511 │            0 │
│ 2026-08-26 17:00:00+00   │  1909988 │ 233451 │            0 │
│ 2026-08-26 18:00:00+00   │  2095407 │      0 │            0 │
│ 2026-08-26 19:00:00+00   │  2092866 │      0 │            0 │
│ 2026-08-26 20:00:00+00   │  2160412 │      0 │            0 │
│ 2026-08-26 21:00:00+00   │  2383117 │      0 │            0 │
│ 2026-08-26 22:00:00+00   │  2666466 │      0 │            0 │
│ 2026-08-26 23:00:00+00   │  2442950 │ 

And the ledger: what an operator has already acknowledged, and what the nightly audits found.

In [4]:
con.sql("""
SELECT run_ts, job, check_name, scope, passed, observed, substr(detail, 1, 110) AS detail
FROM pinned.audit_checks WHERE job = 'operator' OR NOT passed ORDER BY run_ts DESC LIMIT 20
""").show(max_width=200)

┌──────────────────────┬─────────────┬──────────────────────┬──────────────────────┬─────────┬──────────┬──────────────────────────────────────────────────────────────────────────────────────────────┐
│        run_ts        │     job     │      check_name      │        scope         │ passed  │ observed │                                            detail                                            │
│ timestamp with tim…  │   varchar   │       varchar        │       varchar        │ boolean │  int64   │                                           varchar                                            │
├──────────────────────┼─────────────┼──────────────────────┼──────────────────────┼─────────┼──────────┼──────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2026-08-27 08:01:5…  │ operator    │ checksum_failure_a…  │ lake.silver.book_k…  │ true    │   386962 │ from 2026-08-26T16:00:00Z to 2026-08-26T18:00:00Z: capture-kill / queue-full / redpanda-st